<H1>TL_Example2</H1> 
Transmission line example, Complex case or second one on the page. 
power lost due to a lossy line into a resistive load. 
<hr>
<H2>Modification history </H2><br> 
<table>
    <tr>
        <th> Modified </th>
        <th> By </th>
        <th> Reason </th>
    </tr>
    <tr> 
        <td> 08-Mar-26 </td>
        <td> CBL </td> 
        <td> Original </td>
    </tr>
</table>
<hr> 
<H2> References </H2> 
<a href = "https://scikit-rf.readthedocs.io/en/v1.8.0/examples/networktheory/Transmission%20Line%20Losses.html"> Transmission Line Example </a><br> 
<a href = "https://www.fars.k6ya.org/docs/K6OIK-A_Transmission_Line_Power_Paradox_and_Its_Resolution.pdf"> [1] Steve Stearns (K6OIK), A Transmission Line Power Paradox and Its Resolution, ARRL Pacificon Antenna Seminar, Santa Clara, CA October 10-12, 201 </a>
<H2> Notes </H2>

In [1]:
# imports - package libraries etc. 
import numpy as np
import matplotlib.pyplot as plt
import skrf as rf
from skrf import Network
from skrf import Frequency
from skrf.circuit import Circuit
from skrf.tlineFunctions import INF
#
# skrf figure styling
rf.stylely()

Assume a coaxial line, 
<a href="https://thewireman.com"> Wireman #551 450-Ohm </a> loaded with a complex load $Z_L - R_L + j X_L$ with the following
parameters. 
<ul> 
    <li> line length 100 feet </li>
    <li> frequency 1.83 MHz</li>
    <li> attenuation constant: $\alpha = 0.095$ dB/100 feet</li>
    <li> coaxial relative permittivity: $\epsilon_r = 1.194 $ and Velocity Factor = 0.915</li>
    <li> real part of the characteristic impedence: $R_0 = \mathbb{R}[Z_0] = 402.75 \Omega$ </li>
    <li> Load resistance: $R_L = 4.5 \Omega$</li>
    <li> Load reactance: $X_L = -1673 \Omega$ </li>
</ul>

In [2]:
Z_L = 4.5 - 1673j
R_0 = 402.75
freq = rf.Frequency(1.83, npoints=1, unit='MHz')
VF = 0.915
L = rf.mathFunctions.feet_2_meter(100)

The propigation constant of the transmission line is: $\gamma = \alpha + j\beta$ with $\beta = {\omega \over c} \sqrt{\epsilon_r}$

In [3]:
alpha = rf.mathFunctions.db_2_np(rf.mathFunctions.db_per_100feet_2_db_per_100meter((0.095)/100)) #N/m
beta = freq.w/rf.constants.c/VF
gamma = alpha + 1j*beta
print('gamma: ', gamma)

gamma:  [0.00035883+0.0419169j]


However the tansmission line characteristic reactance is not given in the problem parameters and
must be determined. It can be approximated form a high-frequency, low-loss approximation.

$Z_0 = R_0 + jX_0 \approx R_0 - j {\alpha \over \beta} R_0$

or

$Z_0 \approx -{\alpha \over \beta R_0}$

In [4]:
X_0 = -alpha/beta * R_0
Z_0 = R_0 + 1j*X_0
print('X_0 = ', X_0)

X_0 =  [-3.44778935]


Now that we have both the characteristic impedance and the propagation constant of the line, the reflections coefficients, input
impedence and total loss can be deduced.

In [20]:
print('Gamma at load: ', np.abs(rf.tlineFunctions.zl_2_Gamma0(Z_0, Z_L)))
print('Gamma at input: ', np.abs(rf.tlineFunctions.zl_2_Gamma_in(Z_0, Z_L, theta = gamma*L)))
print('SWR at load: ', rf.tlineFunctions.Gamma0_2_swr(rf.tlineFunctions.zl_2_Gamma0(Z_0, Z_L)))
print('SWR at input: ', rf.tlineFunctions.Gamma0_2_swr(rf.tlineFunctions.zl_2_Gamma_in(Z_0, Z_L, theta = gamma*L)))
print('Input Impedance: ', rf.tlineFunctions.input_impedance_at_theta(Z_0, Z_L, theta = gamma*L), 'Ohm')
total_loss_db = rf.mathFunctions.mag_2_db10(np.abs(rf.tlineFunctions.zl_2_total_loss(z0=Z_0, zl=Z_L, theta=gamma*L)))
print('Total loss: ', total_loss_db, ' dB')

Gamma at load:  [0.99489308]
Gamma at input:  [0.97336654]
SWR at load:  [390.62580446]
SWR at input:  [74.09350335]
Input Impedance:  [5.25687961-22.99765972j] Ohm
Total loss:  [13.27540882]  dB
